# Model A · Predict a short-term fall in measured oxygen saturation

At time *t*, use the preceding 30 seconds of heart rate and peripheral oxygen saturation (SpO₂) to predict whether reference SpO₂ will fall at least four percentage points below its recent baseline and stay there for five elapsed seconds within the next 15 seconds. This engineering endpoint describes a measured oxygen trend. Prospective swimmer blackout and drowning outcomes require different labels and validation on the chest-mounted device.

**Run on Kaggle.** Import this notebook, use a CPU session and Run All. With Internet enabled, it downloads about 9 MB of public human data with checksums into a temporary cache. With Internet disabled, attach the unpacked originals for the enabled sources (or place them below `MODEL_A_DATA_ROOT`). Set `QUICK=True` in the next cell for a smaller participant subset. A missing requested file stops the run and identifies the source. Use Python, CPU (no GPU/TPU), and Internet enabled in Kaggle notebook settings. No secrets or API keys are needed. Local runs need Python 3.10+, NumPy, pandas, openpyxl, requests, scikit-learn, joblib, threadpoolctl and matplotlib. The first cell reports missing packages; do not silently upgrade an existing Kaggle environment. The final cell lists exported files.

| Primary data | Subjects | Input HR and target oxygen | Step and limitation |
|---|---:|---|---|
| [Dry intermittent apnea](https://zenodo.org/records/21701010) | 80 | Polar H10 chest-strap HR and Nonin finger SpO₂ | One second, six-minute dry supine protocol. Use the source workbook's FC and SpO2 sheets. |
| [HAPP altitude experiment](https://doi.org/10.6084/m9.figshare.29947679) | 20 | Chest-electrode HR and finger SpO₂ | One second. Authors forward-filled some missing values without an imputation flag. [Methods](https://pmc.ncbi.nlm.nih.gov/articles/PMC12886808/) |
| [UW controlled hypoxemia](https://github.com/ubicomplab/oximetry-phone-cam-data) | 6 | One Nellcor reference oximeter's Pulse 2 and SpO2 2 | One second. Pulse rate is an optical proxy for HR and can share artifacts with the label. Only reference CSVs are needed. |

The benchmark compares HR-only, oxygen-only and combined logistic regression, a small gradient-boosted tree model, and a linear oxygen-trend rule on exactly the same eligible anchors. Device depth, motion, PPG quality and emergency outcomes have separate data requirements.

**Recommendation and scope.** This is an offline research comparison. Start with qualified measurements and a separate current-oxygen concern algorithm; this predictor must not cancel concern about oxygen that is already low. Compare oxygen-only against combined inputs before claiming added HR value. The existing 2026 development run detected 0/4 held-out episodes with the learned models and 2/4 with the trend rule (only two event-bearing people). Those results do not establish deployment readiness. Because this split has already been inspected, redesigns informed by its failures need untouched external/future data for new effectiveness claims.

**Downloads without Internet.** Attach the unchanged `Supp database.xlsx` from the linked Zenodo record, all 20 named CSVs from the linked HAPP archive, and the six `data/gt/100001.csv` … `100006.csv` files from the pinned UW repository revision shown below. Keep filenames unchanged; any subfolder under `/kaggle/input` is searched. Set `ALLOW_DOWNLOAD=False`. Camera videos and raw PPG are not needed here. The SHA/MD5 checks deliberately reject changed source versions. Outputs and PNG evaluation figures are written under `/kaggle/working/model_a`; source files are cached under `/kaggle/temp`, avoiding duplicate dataset exports.


In [ ]:
# Dependency preflight: use Kaggle's installed scientific Python environment.
import importlib.util
required = {"numpy":"numpy", "pandas":"pandas", "openpyxl":"openpyxl", "joblib":"joblib",
            "requests":"requests", "sklearn":"scikit-learn", "threadpoolctl":"threadpoolctl", "matplotlib":"matplotlib"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ImportError("Missing packages: " + ", ".join(missing) +
                      ". Enable Internet and install only these packages in a separate cell with %pip install, then restart and Run All.")
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import hashlib, json, os, platform, re, time
import numpy as np
import pandas as pd
import openpyxl, joblib, requests, sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score, precision_recall_curve, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from numpy.lib.stride_tricks import sliding_window_view

def env_flag(name, default):
    return os.getenv(name, str(default)).lower() in {"1", "true", "yes"}

DATA_ROOT = Path(os.getenv("MODEL_A_DATA_ROOT", "/kaggle/input"))
DEFAULT_OUT = "/kaggle/working/model_a" if Path("/kaggle").exists() else "model_a_output"
OUT = Path(os.getenv("MODEL_A_OUTPUT_DIR", DEFAULT_OUT))
OUT.mkdir(parents=True, exist_ok=True)
CACHE = Path(os.getenv("MODEL_A_CACHE", "/kaggle/temp/model_a_sources" if Path("/kaggle").exists()
                        else str(OUT.parent / "model_a_source_cache")))
ALLOW_DOWNLOAD = env_flag("MODEL_A_DOWNLOAD", True)
QUICK = env_flag("MODEL_A_QUICK", False)
SOURCE_HOLDOUT = env_flag("MODEL_A_SOURCE_HOLDOUT", True)
SOURCES = ["dry_apnea", "happ", "uw"]  # remove any unwanted study before Run All
SEED = 2026
HISTORY, BASELINE, HORIZON, DURATION, DROP_PP = 30, 10, 15, 5, 4.0
MAX_UNMATCHED_PER_HOUR = 2.0  # Shared research budget, measured on validation participants.
MAX_WARNING_FRACTION = 0.10  # Prevent a persistent always-on warning from gaming a start-rate budget.
assert 0 <= MAX_WARNING_FRACTION <= 1
assert np.isfinite(MAX_UNMATCHED_PER_HOUR) and MAX_UNMATCHED_PER_HOUR >= 0
TRAIN_STRIDE = 5  # seconds between eligible training anchors; evaluate every eligible second
assert 3 <= BASELINE <= HISTORY and 1 <= DURATION < HORIZON and DROP_PP > 0
print("Studies:", SOURCES, "| Quick:", QUICK, "| Output:", OUT.resolve())
MAKE_PLOTS = env_flag("MODEL_A_PLOTS", True)
MAX_REPLAY_EPISODES = 12  # All four episodes in the inspected full-data test are shown; cap larger future datasets.
FIGURES = OUT / "figures"
FIGURES.mkdir(exist_ok=True)


### Acquire and verify the source files

Dry-apnea and HAPP downloads use their archive MD5 values. UW rows use a pinned Git commit and each original file's Git blob checksum. Attach `Supp database.xlsx`, the HAPP participant CSVs (for example `0609b.csv`), and all six UW reference CSVs (`100001.csv` through `100006.csv`). Unpack archives before attaching. Quick mode takes a deterministic subset, independent of outcomes. To run with only one study, edit `SOURCES`. Each enabled study has enough participants for separate training, validation and test sets.

In [ ]:
HAPP_FILES = [
    ("0609b.csv", 57303830, "0f107f209da5523d8f2c8b9407e54e11"),
    ("0610b.csv", 57303833, "62e41f18b4e20d7c641bba5544c51207"),
    ("0613b.csv", 57303836, "ffb57c13160aec154e392eb3ee2f8d3e"),
    ("0620a.csv", 57303839, "06dbd2713689f3ae3cc406e119c09095"),
    ("0620b.csv", 57303842, "20ad187a4f87abc0a631a20fef8f20f0"),
    ("0623a.csv", 57303845, "0392bfadce426ec6274a8a18699845fa"),
    ("0624a.csv", 57303848, "8a7437600bccd94641428007ae953b46"),
    ("0624b.csv", 57303851, "cdad3a47e0444ffb08091abf97d89392"),
    ("0626a.csv", 57303854, "14b19ecfdc45118a299ffa65119c7aa1"),
    ("0626b.csv", 57303857, "3406d6a0d4f8f3f07e8a7fbecac81ca2"),
    ("0627a.csv", 57303860, "8ba2cb0a6d738bee4fbaa2081038506f"),
    ("0714a.csv", 57303863, "d304ef294a2f4026b1874250a7a037b2"),
    ("0714b.csv", 57303866, "b114142022139784a7607877641d8d82"),
    ("0715a.csv", 57303869, "d25c537a49504549921f5098bec4747e"),
    ("0717b.csv", 57303872, "fd5de165a346d9b7b49f0f82cfac04f6"),
    ("0721b.csv", 57303875, "ddede47bc85bdf0e22cddc8c85cead96"),
    ("0722a.csv", 57303878, "9172d814e5173d179397353dbe046843"),
    ("0722b.csv", 57303881, "ace5f34ee7a9458863d82ab56f552f71"),
    ("0723b.csv", 57303884, "e0f48e773c817274635662331090cf82"),
    ("0728b.csv", 57303887, "36cf7e074f53d879f9fbc1b2a3d6d88f"),
]
UW_FILES = [
    ("100001.csv", "1a82a222775583e7dfa8df9e5061cda897fdde83"),
    ("100002.csv", "d84cc7d0a12435d69b67e47b1f493d5cf0aefb3c"),
    ("100003.csv", "9110ad6ebccbd1f3417fadf81a4e071ea4d7426c"),
    ("100004.csv", "136cc180ff34084ca791521c18ad2183ae10188f"),
    ("100005.csv", "5c63857e43e258fc590619f04b99792411e5d9e1"),
    ("100006.csv", "c6afaf83a83095c0780428f60769c66d3993615e"),
]
UW_COMMIT = "c483ae8fe4cd32b0b9ce412d07299b05279fc58f"
manifest = [("dry_apnea", "Supp database.xlsx",
             "https://zenodo.org/records/21701010/files/Supp%20database.xlsx?download=1",
             "md5", "79fc9abbd6e3e04c52d17df3247751ec")]
manifest += [("happ", name, f"https://ndownloader.figshare.com/files/{fid}", "md5", checksum)
             for name, fid, checksum in HAPP_FILES]
manifest += [("uw", name, f"https://raw.githubusercontent.com/ubicomplab/oximetry-phone-cam-data/{UW_COMMIT}/data/gt/{name}",
              "git_blob", checksum) for name, checksum in UW_FILES]
if not SOURCES or not set(SOURCES) <= {"dry_apnea", "happ", "uw"}:
    raise ValueError("Select at least one supported study in SOURCES.")
manifest = [m for m in manifest if m[0] in SOURCES]
if QUICK:
    happ_names = {name for name, _, _ in HAPP_FILES[:8]}
    manifest = [m for m in manifest if m[0] != "happ" or m[1] in happ_names]

def verified(path, method, expected):
    data = path.read_bytes()
    digest = (hashlib.md5(data).hexdigest() if method == "md5" else
              hashlib.sha1(b"blob " + str(len(data)).encode() + b"\x00" + data).hexdigest())
    return digest == expected

wanted = {m[1].casefold() for m in manifest}
attached = {}
if DATA_ROOT.is_dir():
    for folder, _, names in os.walk(DATA_ROOT):
        for name in names:
            if name.casefold() in wanted:
                attached.setdefault(name.casefold(), []).append(Path(folder, name))

def source_file(item):
    source, name, url, method, digest = item
    cached = CACHE / source / name
    for path in [*attached.get(name.casefold(), []), cached]:
        if path.is_file() and verified(path, method, digest):
            return source, name, path
    if not ALLOW_DOWNLOAD:
        raise FileNotFoundError(f"Attach the original {source}/{name} below {DATA_ROOT} or enable Internet/download.")
    cached.parent.mkdir(parents=True, exist_ok=True)
    last_error = None
    partial = cached.with_suffix(cached.suffix + ".part")
    for attempt in range(3):
        try:
            resp = requests.get(url, timeout=(12, 90), headers={"User-Agent":"Model-A-research-notebook/1.0"})
            resp.raise_for_status()
            partial.write_bytes(resp.content)
            if not verified(partial, method, digest):
                raise ValueError("Checksum mismatch; check source version or attach the pinned original")
            partial.replace(cached)
            return source, name, cached
        except (requests.RequestException, OSError, ValueError) as error:
            last_error = error
            if attempt < 2:
                time.sleep(1 + attempt)
    raise RuntimeError(f"Cannot acquire {source}/{name} from {url}: {last_error}. "
                       "Enable Kaggle Internet or attach the original file.") from last_error

with ThreadPoolExecutor(max_workers=3) as pool:
    files = list(pool.map(source_file, manifest))
print(pd.Series([s for s, _, _ in files]).value_counts().rename("verified files").to_string())

### Preserve subjects, clocks and missing measurements

Each record stays in its original order. Split at clock jumps, duplicated seconds or backward timestamps. Preserve missing values rather than interpolating them. Device-zero placeholders are excluded. Future HR never determines label eligibility. Source identity is used for splitting, training weights and evaluation only.

In [ ]:
def make_record(source, subject, seconds, hr, spo2):
    frame = pd.DataFrame({"second": pd.to_numeric(seconds, errors="coerce"),
                          "hr": pd.to_numeric(hr, errors="coerce"),
                          "spo2": pd.to_numeric(spo2, errors="coerce")})
    frame["hr"] = frame.hr.where(np.isfinite(frame.hr) & frame.hr.gt(0))
    frame["spo2"] = frame.spo2.where(frame.spo2.gt(0) & frame.spo2.le(100))
    return {"source": source, "subject": f"{source}/{subject}", "frame": frame.reset_index(drop=True)}

def dry_records(path):
    workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
    signals = {}
    try:
        for sheet in ("FC", "SpO2"):
            rows = workbook[sheet].iter_rows(values_only=True)
            header = next(rows)
            indexes = [(i, int(pd.to_timedelta(str(h)).total_seconds())) for i, h in enumerate(header)
                       if re.fullmatch(r"\d{1,2}:\d{2}:\d{2}", str(h))]
            if [s for _, s in indexes] != list(range(360)):
                raise ValueError(f"Unexpected elapsed time columns in {sheet}")
            signals[sheet] = {int(row[0]): [row[i] for i, _ in indexes] for row in rows
                              if row[1] in {"AG", "CG"} and row[0] is not None}
    finally:
        workbook.close()
    if signals["FC"].keys() != signals["SpO2"].keys():
        raise ValueError("Unmatched dry-apnea participant IDs")
    ids = sorted(signals["FC"])
    if QUICK:
        ids = sorted(np.random.default_rng(SEED).choice(ids, size=min(24, len(ids)), replace=False))
    return [make_record("dry_apnea", pid, np.arange(360), signals["FC"][pid], signals["SpO2"][pid]) for pid in ids]

records = []
for source, name, path in files:
    if source == "dry_apnea":
        records.extend(dry_records(path))
    else:
        csv = pd.read_csv(path)
        # UW 100004 has a blank first-column header, with valid HH:MM:SS values.
        clock = csv["Time"] if "Time" in csv.columns else csv.iloc[:, 0]
        s = pd.to_timedelta(clock.astype(str).str.strip(), errors="coerce").dt.total_seconds()
        if s.notna().mean() < 0.95 or not s.dropna().between(0, 86400, inclusive="left").all():
            raise ValueError(f"Unrecognized clock in {name}; expected HH:MM:SS")
        hr_col, o2_col = ("Heart rate", "SPO2") if source == "happ" else ("Pulse 2", "SpO2 2")
        records.append(make_record(source, path.stem, s, csv[hr_col], csv[o2_col]))

# Same study-qualified participant goes to exactly one fold before windows exist.
rng = np.random.default_rng(SEED)
split_of = {}
for source in sorted(SOURCES):
    ids = np.array(sorted({r["subject"] for r in records if r["source"] == source}))
    rng.shuffle(ids)
    n_test = max(1, round(0.2 * len(ids)))
    n_val = max(1, round(0.2 * len(ids)))
    if n_test + n_val >= len(ids):
        raise ValueError(f"{source} needs at least three independent participants")
    for group, subset in (("test", ids[:n_test]), ("validation", ids[n_test:n_test+n_val]),
                          ("train", ids[n_test+n_val:])):
        split_of.update({str(pid): group for pid in subset})

record_audit = pd.DataFrame([
    {"source": r["source"], "subject": r["subject"], "split": split_of[r["subject"]],
     "rows": len(r["frame"]), "paired": int(r["frame"][["hr", "spo2"]].notna().all(axis=1).sum())}
    for r in records])
print(record_audit.groupby(["source", "split"]).agg(people=("subject","nunique"), rows=("rows","sum"),
                                                  paired=("paired","sum")).to_string())

### Build the label and features separately

All variants use the same eligible anchors. The HR-only fit uses HR features, while the shared eligibility checks use observed HR and oxygen. Its benchmark therefore describes the joint-monitoring setting.

Freeze the median SpO₂ across seconds *t−9 … t*. A positive example starts above baseline minus four percentage points at *t*, then reaches or falls below that boundary for at least five **elapsed seconds** within *t+1 … t+15*. For example, baseline 97% and values ≤93% at every second from *t+6* to *t+11* qualify. Six one-second observations span five elapsed seconds. These are benchmark settings.

For a fair ablation, every variant uses the same eligible anchors: a complete 30-second paired history, current oxygen above the boundary, and complete future reference SpO₂. The HR-only model's feature vector contains HR alone. This conditional benchmark uses oxygen for eligibility/labels. The future data select evaluation examples offline and never enter a feature vector. A continuously low oxygen level can occur without a new relative drop.

The first qualifying future run for each positive anchor supplies an observed support interval. Overlapping/touching support intervals within a record are merged into one benchmark episode. Its start is the earliest supported onset among its anchors. These episodes depend on the frozen-baseline definition. They serve to account for repeated windows around a fall.

In [ ]:
def history_features(history, prefix):
    """Only the already-available history; each row is one anchor."""
    width = history.shape[1]
    x = np.arange(width, dtype=float) - (width - 1) / 2
    x10 = np.arange(10, dtype=float) - 4.5
    last10 = history[:, -10:]
    return {f"{prefix}_last": history[:, -1], f"{prefix}_median": np.median(history, axis=1),
            f"{prefix}_minimum": np.min(history, axis=1), f"{prefix}_sd": np.std(history, axis=1),
            f"{prefix}_recent_delta": np.median(history[:, -5:], axis=1) - np.median(history[:, :5], axis=1),
            f"{prefix}_slope_10s": (last10 @ x10) / (x10 @ x10),
            f"{prefix}_slope_full": (history @ x) / (x @ x)}

def future_run(future_oxygen, boundary):
    below = future_oxygen <= boundary[:, None]
    runs = sliding_window_view(below, DURATION + 1, axis=1).all(axis=2)
    return runs.any(axis=1), runs.argmax(axis=1) + 1  # first qualifying run, in seconds after t

# Endpoint boundary check: five low samples span four seconds; six span five.
test_oxygen = np.full((2, HORIZON), 97.0)
test_oxygen[0, :DURATION] = 93
test_oxygen[1, -DURATION-1:] = 93
assert future_run(test_oxygen, np.array([93, 93]))[0].tolist() == [False, True]

def segment_windows(segment, record, span):
    if len(segment) < HISTORY + HORIZON:
        return None
    t = segment.second.to_numpy(dtype=float)
    hr = sliding_window_view(segment.hr.to_numpy(dtype=float), HISTORY)[:-HORIZON]
    oxygen = sliding_window_view(segment.spo2.to_numpy(dtype=float), HISTORY + HORIZON)
    past, future = oxygen[:, :HISTORY], oxygen[:, HISTORY:]
    good = np.isfinite(hr).all(axis=1) & np.isfinite(oxygen).all(axis=1)
    if not good.any():
        return None
    hr, past, future = hr[good], past[good], future[good]
    anchor = t[HISTORY - 1 : -HORIZON][good].astype(np.int64)
    boundary = np.median(past[:, -BASELINE:], axis=1) - DROP_PP
    eligible = past[:, -1] > boundary
    if not eligible.any():
        return None
    hr, past, future = hr[eligible], past[eligible], future[eligible]
    anchor, boundary = anchor[eligible], boundary[eligible]
    positive, first = future_run(future, boundary)
    onset = np.where(positive, anchor + first, -1)
    episode = np.full(len(anchor), -1, dtype=int)
    first_onset = np.full(len(anchor), -1, dtype=np.int64)
    event_num, end = -1, -10**12
    for i in np.flatnonzero(positive)[np.argsort(onset[positive], kind="stable")]:
        if onset[i] > end:
            event_num += 1
            start = onset[i]
        end = max(end, onset[i] + DURATION)
        episode[i], first_onset[i] = event_num, start
    features = history_features(hr, "hr") | history_features(past, "spo2")
    frame = pd.DataFrame(features)
    frame["label_baseline_spo2"] = boundary + DROP_PP
    frame["label_boundary_spo2"] = boundary
    # Higher trend_rule scores correspond to a larger anticipated fall (percentage points).
    frame["trend_rule"] = (boundary + DROP_PP -
                           (past[:, -1] + frame.spo2_slope_10s * (HORIZON - DURATION)))
    frame["source"], frame["subject"] = record["source"], record["subject"]
    frame["span"], frame["t"], frame["target"] = span, anchor, positive.astype(int)
    frame["episode"], frame["episode_onset"] = episode, first_onset
    frame["split"] = split_of[record["subject"]]
    return frame

def collect_windows(records):
    parts = []
    for r in records:
        frame = r["frame"]
        # Do not sort clock resets or duplicate seconds into a fictitious uninterrupted trace.
        for span, segment in frame.groupby(frame.second.diff().ne(1).cumsum(), sort=False):
            if segment.second.isna().any():
                continue
            part = segment_windows(segment, r, int(span))
            if part is not None:
                parts.append(part)
    if not parts:
        raise ValueError("No complete eligible anchors under this endpoint and source selection")
    return pd.concat(parts, ignore_index=True)

assert HISTORY >= 10
with threadpool_limits(limits=1):
    windows = collect_windows(records)
EPISODE_KEY = ["subject", "span", "episode"]
HR_FEATURES = [c for c in windows if c.startswith("hr_")]
O2_FEATURES = [c for c in windows if c.startswith("spo2_")]
ALL_FEATURES = HR_FEATURES + O2_FEATURES
support = (windows.groupby(["source", "split"]).agg(eligible_seconds=("t", "size"), positive_anchors=("target", "sum"),
                                                     participants_with_windows=("subject", "nunique")))
counts = (windows[windows.target.eq(1)].drop_duplicates(EPISODE_KEY)
          .groupby(["source", "split"]).size().rename("episodes"))
support = record_audit.groupby(["source","split"]).agg(people=("subject","nunique"), source_rows=("rows","sum")).join(support).join(counts)
people_with_events = (windows.loc[windows.target.eq(1)].groupby(["source", "split"]).subject.nunique()
                      .rename("people_with_events"))
support = support.join(people_with_events).fillna(0).astype(int)
support["eligible_fraction_of_source_rows"] = support.eligible_seconds / support.source_rows
print(support.to_string())
print("Features:", ALL_FEATURES)

### Fit and select thresholds using the development participants

The logistic models have a fixed L2 penalty. The tree model uses shallow trees, full training data and a fixed iteration limit, with internal random early stopping disabled. Training takes one eligible anchor per five-second grid interval and gives every study equal total weight, then every training participant equal weight within that study. Validation and test retain every eligible second. Scaling is fitted on training only.

Compare every variant under the same validation budget of **two unmatched alert starts per eligible hour**, with at most **10% of eligible validation time in warning**. Both settings are adjustable engineering choices. The time cap prevents one persistent alert from meeting the start-rate budget in a long recording. For each validation score distribution, search 301 quantiles across the full range, 101 more in the highest 1%, and an option that never alerts. From the candidates within budget, choose the one that detects the most distinct episodes. Break ties by fewer unmatched starts, fewer seconds spent in warning, and then the higher threshold. If none detects an episode within budget, disable alerts. Held-out rates can exceed the validation budget. Inspect both rates and the warning-time fraction. The study-balanced training mix makes external calibration a separate experiment.

The constant comparator uses the unweighted prevalence of the same thinned training anchors and ignores its inputs. It is not a physiological model. Compare its probability losses with the source/person-weighted fits, whose training distribution differs from observed window frequency. `predict_proba` is a research score until calibration and transport to the intended population are established. We do not fit an extra calibrator to these few independent events.


In [ ]:
CONSTANT = "Constant training prevalence"
MODEL_COLS = {"HR logistic": HR_FEATURES, "SpO2 logistic": O2_FEATURES,
              "HR + SpO2 logistic": ALL_FEATURES, "HR + SpO2 boosted": ALL_FEATURES, CONSTANT: O2_FEATURES}
RULE = "SpO2 trend rule"

def training_sample(train):
    # A record-local grid keeps cadence independent of labels and boundary exclusions.
    thin = train[train.t.mod(TRAIN_STRIDE).eq(0)].copy()
    if thin.target.nunique() < 2:
        raise ValueError("Training needs observations from both endpoint classes. Use the full data or more sources.")
    n_people = thin.groupby("source").subject.transform("nunique")
    per_person = thin.groupby("subject").subject.transform("size")
    w = 1.0 / (n_people * per_person)
    return thin, (w / w.mean()).to_numpy()

def fit_models(train, variants):
    data, weights = training_sample(train)
    fitted = {}
    for name in variants:
        cols = MODEL_COLS[name]
        if name == CONSTANT:
            model = DummyClassifier(strategy="prior")
            model.fit(data[cols], data.target)
        elif name.endswith("boosted"):
            model = HistGradientBoostingClassifier(max_iter=45 if QUICK else 70, max_leaf_nodes=7,
                                                   learning_rate=0.07, min_samples_leaf=35,
                                                   l2_regularization=2, early_stopping=False, random_state=SEED)
            model.fit(data[cols], data.target, sample_weight=weights)
        else:
            model = make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=500, random_state=SEED))
            model.fit(data[cols], data.target, standardscaler__sample_weight=weights,
                      logisticregression__sample_weight=weights)
        fitted[name] = model
    return fitted

def predict_scores(models, data):
    scores = {name: model.predict_proba(data[MODEL_COLS[name]])[:, 1] for name, model in models.items()}
    scores[RULE] = data.trend_rule.to_numpy()
    return scores

def alert_context(frame):
    # Rows are in recording order. Only consecutive eligible anchors sustain an alert.
    continuous = ((frame.subject == frame.subject.shift()) & (frame.span == frame.span.shift())
                  & frame.t.diff().eq(1)).to_numpy()
    early = (frame.target.eq(1) & frame.t.lt(frame.episode_onset)).to_numpy()
    episode_ids = frame.groupby(EPISODE_KEY, sort=False).ngroup().to_numpy()
    return continuous, early, episode_ids

def alert_outcomes(high, context):
    continuous, early, episode_ids = context
    start = high & ~(np.r_[False, high[:-1]] & continuous)
    timely_idx = np.flatnonzero(start & early)
    _, first = np.unique(episode_ids[timely_idx], return_index=True)
    credited_idx = timely_idx[first]
    unmatched = int(start.sum()) - len(credited_idx)
    return start, credited_idx, unmatched

def threshold_on_validation(frame, score):
    if frame.target.nunique() < 2:
        return None
    context = alert_context(frame)
    hour_count = len(frame) / 3600
    q = np.unique(np.r_[np.linspace(0, 1, 301), np.linspace(.99, 1, 101)])
    candidates = np.unique(np.quantile(score, q))
    best, threshold = (0, 0, 0, float("inf")), None  # A genuine no-alert option.
    for value in candidates:
        high = score >= value
        _, credited, unmatched = alert_outcomes(high, context)
        if unmatched / hour_count > MAX_UNMATCHED_PER_HOUR or high.mean() > MAX_WARNING_FRACTION:
            continue
        rank = (len(credited), -unmatched, -int(high.sum()), value)
        if rank > best:
            best, threshold = rank, float(value)
    return threshold

train = windows[windows.split.eq("train")].reset_index(drop=True)
validation = windows[windows.split.eq("validation")].reset_index(drop=True)
test = windows[windows.split.eq("test")].reset_index(drop=True)
if validation.empty or test.empty:
    raise ValueError("No eligible validation or test anchors. Check source coverage and quick mode.")
assert not set(train.subject) & (set(validation.subject) | set(test.subject))
assert not set(validation.subject) & set(test.subject)
with threadpool_limits(limits=1):
    models = fit_models(train, MODEL_COLS)
    val_scores = predict_scores(models, validation)
thresholds = {name: threshold_on_validation(validation, score) for name, score in val_scores.items()}
if any(v is None for v in thresholds.values()):
    print("A blank threshold means disabled: insufficient validation classes or no detected episode within budget.")
print(pd.Series(thresholds, name="validation threshold").to_string())

### Evaluate held-out participants and count alert episodes

An alert opens on a threshold crossing and stays open through consecutive eligible high-score seconds. Missing/ineligible seconds break a run. A new alert can receive credit for its own anchor's positive reference episode if it opens before the episode's earliest supported onset. Credit each episode once. Unmatched starts include duplicate or late starts. The denominator for alerts per hour is **eligible observed prediction time**, reported beside the count. Also inspect the fraction of eligible time in warning: a threshold that stays high throughout a recording can open very few alerts.

Averaging many adjacent windows can give a misleading impression of sample size. Read the test results by source, with the participant count, positive-anchor count and unique episode count. Average precision and AUROC are left blank when a source lacks either class. Brier scores apply only to the fitted classifiers' raw probability outputs. Deployment calibration and false alerts per real swim-hour require representative chest recordings.

In [ ]:
def summarize(frame, score, cutoff, is_probability):
    y = frame.target.to_numpy()
    episode_count = len(frame.loc[frame.episode.ge(0), EPISODE_KEY].drop_duplicates())
    both = len(np.unique(y)) == 2
    row = {"subjects": frame.subject.nunique(), "people_with_events": frame.loc[frame.target.eq(1), "subject"].nunique(),
           "eligible_seconds": len(frame),
           "eligible_hours": len(frame)/3600, "positive_anchors": int(y.sum()),
           "prevalence": float(y.mean()), "reference_episodes": episode_count,
           "AP": average_precision_score(y, score) if both else np.nan,
           "AUROC": roc_auc_score(y, score) if both else np.nan,
           "Brier": brier_score_loss(y, score) if is_probability else np.nan,
           "log_loss": log_loss(y, np.column_stack([1-np.asarray(score), score]), labels=[0,1]) if is_probability else np.nan}
    high = (np.asarray(score) >= cutoff) if cutoff is not None else np.zeros(len(frame), dtype=bool)
    start, credited_idx, unmatched = alert_outcomes(high, alert_context(frame))
    credited = frame.iloc[credited_idx]
    matched, starts = len(credited), int(start.sum())
    leads = credited.episode_onset - credited.t
    tp = int(np.sum(high & (y == 1)))
    row.update({"cutoff": cutoff, "alerts_enabled": cutoff is not None, "window_precision": tp/high.sum() if high.any() else np.nan,
                "window_recall": tp/y.sum() if y.any() else np.nan,
                "warning_fraction": float(high.mean()), "alert_starts": starts,
                "matched_episodes": matched, "missed_episodes": episode_count - matched,
                "episode_recall": matched/episode_count if episode_count else np.nan,
                "unmatched_starts": unmatched,
                "unmatched_starts_per_eligible_hour": unmatched/(len(frame)/3600),
                "median_lead_seconds": float(leads.median()) if matched else np.nan})
    return row

def evaluation_rows(data, scores, cutoff_by_model, experiment, by_source=True):
    out = []
    groups = [("all", np.ones(len(data), dtype=bool))]
    if by_source:
        groups += [(str(s), data.source.eq(s).to_numpy()) for s in sorted(data.source.unique())]
    for source, take in groups:
        for name, scores_for_model in scores.items():
            vals = summarize(data.loc[take].reset_index(drop=True), scores_for_model[take],
                             cutoff_by_model[name], name != RULE)
            out.append({"experiment": experiment, "source": source, "model": name, **vals})
    return out

with threadpool_limits(limits=1):
    test_scores = predict_scores(models, test)
rows = evaluation_rows(validation, val_scores, thresholds, "validation")
rows += evaluation_rows(test, test_scores, thresholds, "held-out people")
results = pd.DataFrame(rows)
pooled_test = results[(results.experiment == "held-out people") & (results.source == "all")].iloc[0]
print(f"Held-out evidence: {pooled_test.reference_episodes} distinct episodes among "
      f"{pooled_test.people_with_events} positive people, {pooled_test.subjects} test people total. "
      "Each missed episode can change the recall substantially. Inspect source counts below.")
shown = ["source", "model", "subjects", "people_with_events", "positive_anchors", "reference_episodes", "AP", "AUROC", "Brier",
         "alert_starts", "matched_episodes", "eligible_hours", "unmatched_starts_per_eligible_hour",
         "episode_recall", "warning_fraction", "alerts_enabled", "median_lead_seconds"]
print(results.loc[results.experiment.eq("held-out people"), results.columns.intersection(shown, sort=False)]
      .round(3).to_string(index=False))

### Does the model transfer to an unseen study?

Leave one entire study out of training and threshold selection. Fit on the training participants of the remaining studies and select thresholds on their validation participants under the same alert budget. The main transfer result evaluates **every participant in the omitted study**, maximizing independent test coverage. A second result uses only its original test people so you can compare directly with the pooled fit on the same examples. Transfer remains a comparison across dry apnea, altitude, and laboratory hypoxemia. Measuring performance in swimmers requires a separate aquatic test.

In [ ]:
transfer_rows = []
if SOURCE_HOLDOUT and len(SOURCES) > 1:
    with threadpool_limits(limits=1):
        for held in sorted(SOURCES):
            other_train = train[~train.source.eq(held)]
            other_val = validation[~validation.source.eq(held)].reset_index(drop=True)
            held_all = windows[windows.source.eq(held)].reset_index(drop=True)
            held_test = test[test.source.eq(held)].reset_index(drop=True)
            if held_all.empty or other_val.empty or other_train[other_train.t.mod(TRAIN_STRIDE).eq(0)].target.nunique() < 2:
                print("Insufficient training/validation/test support for held-out study:", held)
                continue
            transfer_models = fit_models(other_train, MODEL_COLS)
            ref_scores = predict_scores(transfer_models, other_val)
            cutoffs = {n: threshold_on_validation(other_val, s) for n, s in ref_scores.items()}
            assert not set(held_all.subject) & (set(other_train.subject) | set(other_val.subject))
            for label, held_frame in [("held-out study (all people)", held_all),
                                      ("held-out study (matched test subset)", held_test)]:
                if held_frame.empty:
                    continue
                new_rows = evaluation_rows(held_frame, predict_scores(transfer_models, held_frame),
                                           cutoffs, label, by_source=False)
                transfer_rows.extend(dict(row, source=held) for row in new_rows)
    if transfer_rows:
        print(pd.DataFrame(transfer_rows).query('experiment == "held-out study (all people)"')[["source","model","subjects","people_with_events","reference_episodes","AP","episode_recall","warning_fraction","unmatched_starts_per_eligible_hour"]]
              .round(3).to_string(index=False))
else:
    print("Study holdouts disabled or only one source enabled.")

### Inspect episodes, alert burden, calibration and source transfer

Figures below complement the tables; high window AUROC does not establish useful event warning. An episode earns at most one timely warning. The operating curves show **unmatched alert starts per eligible hour**, not confirmed false drowning alarms per swim-hour. Displayed curves are restricted to the declared warning-time budget; full curves are exported. Held-out curves are descriptive, never used to reselect the marked validation threshold.

Exports include one row per eligible anchor with every model score/alarm, one row per model/reference episode including misses, and every alert start with its credited status. Coverage is **benchmark eligibility** (complete paired history/future plus endpoint rules), not validated live sensor uptime. Source-level event denominators are always shown.

Calibration plots include bin support and a score histogram. Brier loss reflects both discrimination and calibration; a low loss with rare positives can be achieved by a nearly constant score. No post-hoc calibrator is fitted. Four events across two people cannot support a convincing probability-calibration claim or narrow uncertainty intervals. Overlapping windows are not independent observations.

Replay plots include all reference episodes up to the displayed cap and the first two unmatched oxygen-only alert starts. The candidate alarm only uses past data; future oxygen and labels are shown here solely for offline inspection. Gray intervals indicate absent eligible predictions, not zero physiological risk.


In [ ]:
# Save inspectable prediction and event tables before drawing figures.
anchor_exports, event_exports, alert_exports = [], [], []
for split_name, frame, scores_by_name in [("validation", validation, val_scores), ("test", test, test_scores)]:
    exported = frame[["source", "subject", "span", "t", "target", "episode", "episode_onset", "label_baseline_spo2", "label_boundary_spo2"]].copy()
    exported["split"] = split_name
    context = alert_context(frame)
    episodes = frame.loc[frame.target.eq(1), ["source", *EPISODE_KEY, "episode_onset"]].drop_duplicates(EPISODE_KEY)
    for name, values in scores_by_name.items():
        cutoff = thresholds[name]
        high = np.asarray(values) >= cutoff if cutoff is not None else np.zeros(len(frame), bool)
        starts, credit_idx, _ = alert_outcomes(high, context)
        exported["score__" + name] = values
        exported["warning__" + name] = high
        matched = frame.iloc[credit_idx][EPISODE_KEY + ["t"]].rename(columns={"t":"warning_start"})
        event_table = episodes.merge(matched, on=EPISODE_KEY, how="left")
        event_table["detected"] = event_table.warning_start.notna()
        event_table["lead_seconds"] = event_table.episode_onset - event_table.warning_start
        event_table["model"], event_table["split"] = name, split_name
        event_exports.append(event_table)
        start_idx = np.flatnonzero(starts)
        start_table = frame.iloc[start_idx][["source", "subject", "span", "t", "target", "episode"]].copy()
        start_table["credited_to_episode"] = np.isin(start_idx, credit_idx)
        start_table["model"], start_table["split"] = name, split_name
        alert_exports.append(start_table)
    anchor_exports.append(exported)
pd.concat(anchor_exports, ignore_index=True).to_csv(OUT/"anchor_predictions.csv.gz", index=False)
event_table = pd.concat(event_exports, ignore_index=True)
alert_table = pd.concat(alert_exports, ignore_index=True)
event_table.to_csv(OUT/"event_matches.csv", index=False)
alert_table.to_csv(OUT/"alert_starts.csv", index=False)

def operating_rows(frame, scores, split_name):
    context = alert_context(frame)
    count = len(frame.loc[frame.target.eq(1), EPISODE_KEY].drop_duplicates())
    rows = []
    for name, values in scores.items():
        grid = np.unique(np.quantile(values, np.r_[np.linspace(0, 1, 101), np.linspace(.99, 1, 51)]))
        if thresholds[name] is not None:
            grid = np.unique(np.r_[grid, thresholds[name]])
        for threshold in [None, *grid]:
            high = values >= threshold if threshold is not None else np.zeros(len(frame), bool)
            starts, credited, unmatched = alert_outcomes(high, context)
            rows.append({"split":split_name, "model":name, "threshold":threshold,
                         "episode_recall":len(credited)/count if count else np.nan,
                         "unmatched_starts_per_eligible_hour":unmatched/(len(frame)/3600),
                         "warning_fraction":float(high.mean()), "episodes":count,
                         "matched_episodes":len(credited), "alert_starts":int(starts.sum())})
    return rows

operating = pd.DataFrame(operating_rows(validation, val_scores, "validation") + operating_rows(test, test_scores, "test"))
operating.to_csv(OUT/"operating_curves.csv", index=False)
calibration_rows = []
for scope in ["all", *sorted(test.source.unique())]:
    take = np.ones(len(test), bool) if scope == "all" else test.source.eq(scope).to_numpy()
    for name, values in test_scores.items():
        if name == RULE:
            continue
        group = pd.DataFrame({"score":values[take], "y":test.target.to_numpy()[take], "subject":test.subject.to_numpy()[take]})
        # Fixed probability bins keep the diagonal meaningful; report empty/rare support honestly.
        group["bin"] = pd.cut(group.score, np.linspace(0,1,11), include_lowest=True)
        for label, part in group.groupby("bin", observed=True):
            calibration_rows.append({"source":scope, "model":name, "bin":str(label),
                                     "mean_score":part.score.mean(), "observed_frequency":part.y.mean(),
                                     "anchors":len(part), "positive_anchors":int(part.y.sum()),
                                     "people":part.subject.nunique(), "event_people":part.loc[part.y.eq(1),"subject"].nunique()})
calibration_table = pd.DataFrame(calibration_rows)
calibration_table.to_csv(OUT/"calibration_bins.csv", index=False)

def finish_figure(fig, name):
    fig.savefig(FIGURES/name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

if MAKE_PLOTS:
    plt.rcParams.update({"font.size":9, "axes.spines.top":False, "axes.spines.right":False})
    colors = {name:plt.get_cmap("tab10")(i) for i,name in enumerate(test_scores)}
    audit = support.reset_index()
    labels = audit.source + "/" + audit["split"]
    fig, axes = plt.subplots(2,2,figsize=(13,8))
    for ax, column, title, divisor in zip(axes.flat, ["people", "episodes", "eligible_seconds", "eligible_fraction_of_source_rows"],
                                         ["Participants", "Distinct episodes", "Eligible hours", "Eligible anchors / source rows"], [1,1,3600,1]):
        ax.bar(np.arange(len(audit)), audit[column]/divisor)
        ax.set_xticks(np.arange(len(audit)), labels, rotation=65, ha="right")
        ax.set_title(title)
    fig.suptitle("Data support and benchmark eligibility (not live sensor coverage)")
    fig.tight_layout()
    finish_figure(fig, "01_support_and_eligibility.png")

    scopes = ["all", *sorted(test.source.unique())]
    fig, axes = plt.subplots(1,len(scopes),figsize=(4*len(scopes),4), squeeze=False)
    for ax, scope in zip(axes.flat, scopes):
        take = np.ones(len(test), bool) if scope == "all" else test.source.eq(scope).to_numpy()
        y = test.target.to_numpy()[take]
        events = len(test.loc[take & test.target.eq(1), EPISODE_KEY].drop_duplicates())
        if len(np.unique(y)) < 2:
            ax.text(.5,.5,"No two-class support",ha="center",transform=ax.transAxes)
        else:
            for name, values in test_scores.items():
                if name == CONSTANT:
                    continue  # A constant ranker has only the prevalence operating point.
                precision, recall, _ = precision_recall_curve(y, values[take])
                # sklearn returns descending recall; post steps preserve AP's stepwise convention.
                ax.step(recall,precision,where="post",label=name,color=colors[name])
            ax.axhline(y.mean(),color=colors[CONSTANT],ls=":",label="Constant / test prevalence")
        ax.set(xlabel="Window recall",ylabel="Window precision",xlim=(0,1),ylim=(0,1),title=f"{scope}: {events} episodes")
    handles, labels = axes[0,0].get_legend_handles_labels()
    fig.legend(handles,labels,loc="lower center",ncol=3,bbox_to_anchor=(.5,-.15))
    fig.tight_layout()
    finish_figure(fig, "02_precision_recall.png")

    fig, axes = plt.subplots(1,2,figsize=(13,5))
    for ax, split_name in zip(axes,["validation","test"]):
        for name in test_scores:
            group = operating[(operating.split == split_name) & (operating.model == name) &
                              (operating.warning_fraction <= MAX_WARNING_FRACTION)].sort_values("unmatched_starts_per_eligible_hour")
            ax.plot(group.unmatched_starts_per_eligible_hour,group.episode_recall,".-",ms=3,color=colors[name],label=name,alpha=.75)
            row = results[(results.experiment == ("validation" if split_name == "validation" else "held-out people")) &
                          (results.source == "all") & (results.model == name)].iloc[0]
            ax.scatter(row.unmatched_starts_per_eligible_hour,row.episode_recall,s=65,marker="D",color=colors[name],edgecolor="black",zorder=3)
        ax.axvline(MAX_UNMATCHED_PER_HOUR,color="black",ls=":")
        ax.set(xlabel="Unmatched alert starts / eligible hour",ylabel="Episodes detected / episodes",ylim=(-.03,1.03),
               xlim=(0,max(5,2*MAX_UNMATCHED_PER_HOUR)),title=f"{split_name}: diamonds use validation thresholds")
    fig.suptitle(f"Operating curves: warning fraction ≤ {MAX_WARNING_FRACTION:.0%}; full range in operating_curves.csv")
    axes[1].legend(loc="center left",bbox_to_anchor=(1, .5))
    fig.tight_layout()
    finish_figure(fig,"03_event_operating_curves.png")

    fig, axes = plt.subplots(1,2,figsize=(12,4))
    for name, values in test_scores.items():
        if name == RULE: continue
        group = calibration_table[(calibration_table.source == "all") & (calibration_table.model == name)]
        axes[0].plot(group.mean_score,group.observed_frequency,"o-",label=name,color=colors[name])
        axes[1].hist(values,bins=np.linspace(0,1,31),histtype="step",label=name,color=colors[name])
    axes[0].plot([0,1],[0,1],"k:")
    axes[0].set(xlabel="Mean predicted event score",ylabel="Observed positive-window frequency",xlim=(0,1),ylim=(0,1),title="Descriptive calibration; bin counts exported")
    axes[1].set(xlabel="Predicted event score",ylabel="Eligible anchors (log scale)",yscale="log",title="Score distribution")
    axes[1].legend(loc="center left",bbox_to_anchor=(1,.5))
    fig.tight_layout()
    finish_figure(fig,"04_calibration_and_scores.png")

    tested = event_table[event_table.split.eq("test")]
    fig, axes = plt.subplots(1,2,figsize=(12,4))
    for i,name in enumerate(test_scores):
        group = tested[tested.model.eq(name)]
        leads = group.loc[group.detected,"lead_seconds"].to_numpy()
        axes[0].scatter(leads,np.full(len(leads),i),color=colors[name],s=45)
        axes[1].barh(i,int(group.detected.sum()),color=colors[name])
        axes[1].barh(i,int((~group.detected).sum()),left=int(group.detected.sum()),color="lightgray")
    axes[0].set(yticks=np.arange(len(test_scores)),yticklabels=list(test_scores),xlabel="Seconds before earliest reference onset",title="Lead time for detected episodes only")
    axes[1].set(yticks=np.arange(len(test_scores)),yticklabels=[],xlabel="Episodes: colored=detected; gray=missed",title="Misses remain in the denominator")
    fig.tight_layout()
    finish_figure(fig,"05_lead_time_and_misses.png")

    summary = pd.concat([results,pd.DataFrame(transfer_rows)],ignore_index=True)
    displayed = summary[summary.source.ne("all") & summary.experiment.isin(["held-out people","held-out study (matched test subset)"])].copy()
    fig,axes = plt.subplots(1,3,figsize=(16,5))
    for ax,metric,title in zip(axes,["episode_recall","unmatched_starts_per_eligible_hour","warning_fraction"],
                               ["Event recall", "Unmatched starts / eligible hour", "Warning-time fraction"]):
        table=displayed.pivot(index=["source","experiment"],columns="model",values=metric)
        table.plot.bar(ax=ax,legend=False,color=[colors[n] for n in table.columns])
        ax.set(title=title,xlabel="Same test participants; source excluded only in transfer fit")
        ax.tick_params(axis="x",rotation=75)
    handles,labels=axes[0].get_legend_handles_labels()
    fig.legend(handles,labels,loc="upper center",ncol=3,bbox_to_anchor=(.5,1.08))
    fig.tight_layout()
    finish_figure(fig,"06_source_transfer.png")

    # Replays preserve original order and span identities, including data outside eligible anchors.
    raw_spans = {}
    for record in records:
        for span,segment in record["frame"].groupby(record["frame"].second.diff().ne(1).cumsum(),sort=False):
            raw_spans[(record["subject"],int(span))]=segment
    replay_cases=[]
    for row in test.loc[test.target.eq(1),["subject","span","episode","episode_onset"]].drop_duplicates(EPISODE_KEY).head(MAX_REPLAY_EPISODES).itertuples(index=False):
        replay_cases.append((row.subject,row.span,row.episode_onset,f"Reference episode {row.episode}"))
    unmatched = alert_table[(alert_table.split == "test") & (alert_table.model == "SpO2 logistic") & ~alert_table.credited_to_episode].head(2)
    replay_cases += [(r.subject,r.span,r.t,"Unmatched oxygen-only alert") for r in unmatched.itertuples(index=False)]
    for index,(subject,span,center,label) in enumerate(replay_cases):
        raw=raw_spans[(subject,int(span))]
        raw=raw[raw.second.between(center-45,center+30)]
        take=(test.subject.eq(subject)&test.span.eq(span)&test.t.between(center-45,center+30)).to_numpy()
        selected=test.loc[take]
        fig,axes=plt.subplots(4,1,figsize=(12,8),sharex=True)
        axes[0].plot(raw.second,raw.spo2,label="Released reference SpO2")
        axes[0].plot(selected.t, selected.label_baseline_spo2, color="gray", ls=":", label="Each anchor's recent baseline")
        axes[0].plot(selected.t, selected.label_boundary_spo2, color="tab:purple", ls=":", label="Each anchor's future-event boundary")
        axes[0].legend(loc="lower left", fontsize=8)
        axes[0].set_ylabel("SpO2 (%)")
        axes[1].plot(raw.second,raw.hr,color="tab:red")
        axes[1].set_ylabel("HR / pulse (bpm)")
        for name in ["SpO2 logistic","HR + SpO2 logistic"]:
            values=test_scores[name][take]
            plot_values = pd.Series(values, index=selected.t).reindex(raw.second).to_numpy()
            axes[2].plot(raw.second,plot_values,label=name,color=colors[name])
            cutoff=thresholds[name]
            if cutoff is not None: axes[2].axhline(cutoff,color=colors[name],ls=":",alpha=.7)
        axes[2].set_ylabel("Forecast score")
        for j,name in enumerate(["SpO2 logistic","HR + SpO2 logistic",RULE]):
            cutoff=thresholds[name]
            high=test_scores[name][take]>=cutoff if cutoff is not None else np.zeros(len(selected),bool)
            axes[3].scatter(selected.t[high],np.full(int(high.sum()),j),marker="|",s=90,color=colors[name])
        axes[3].set(yticks=[0,1,2],yticklabels=["O2 warning","HR+O2 warning","Trend warning"],xlabel="Original recording seconds")
        eligible_times=set(selected.t)
        for ax in axes:
            ax.axvline(center,color="black",ls="--",alpha=.6)
            for t in raw.second:
                if t not in eligible_times: ax.axvspan(t-.5,t+.5,color="gray",alpha=.07,lw=0)
        axes[2].legend(loc="upper left")
        fig.suptitle(f"{subject} | {label} | dashed = onset/alert; gray = prediction ineligible")
        fig.tight_layout()
        finish_figure(fig,f"07_replay_{index+1:02d}.png")
    if not replay_cases: print("No reference events or unmatched oxygen-only starts to replay in this split.")
    print("Evaluation figures:", len(list(FIGURES.glob("*.png"))), "|", FIGURES.resolve())
else:
    print("Plot rendering disabled; evaluation CSVs still exported.")


### Export fitted models and their scoring contract

All models are development fits with thresholds chosen on validation. The test results are descriptive. The output folder contains trusted-local scikit-learn model objects, logistic coefficients, source checksums, assignment of participants to splits, event/coverage counts, and pooled and transfer results. Only load a joblib file you trust. The example function below scores one observed history after the external measurement-validity checks; quality gating, calibration on the chest hardware, and the depth/motion fusion policy are separate steps.

In [ ]:
weights_rows = []
for name, model in models.items():
    if name.endswith("logistic"):
        scaler, logit = model.steps[0][1], model.steps[1][1]
        weights_rows.append({"model": name, "feature": "(intercept)", "coefficient": logit.intercept_[0]})
        for feature, mean, scale, beta in zip(MODEL_COLS[name], scaler.mean_, scaler.scale_, logit.coef_[0]):
            weights_rows.append({"model": name, "feature": feature, "training_mean": mean,
                                 "training_scale": scale, "coefficient": beta})
settings = {"purpose":"future relative reference SpO2 drop; research benchmark", "seed": SEED,
            "sources": SOURCES, "quick": QUICK, "history_samples_1hz": HISTORY, "baseline_samples": BASELINE,
            "drop_percentage_points": DROP_PP, "future_horizon_s": HORIZON, "duration_elapsed_s": DURATION,
            "boundary_operator": "<=", "current_eligibility": "last SpO2 above frozen baseline minus drop",
            "training_stride_s": TRAIN_STRIDE, "validation_test_stride_s": 1,
            "threshold_selection":"validation grid: maximize timely distinct episodes within unmatched-start budget; ties use fewer unmatched starts, fewer warning seconds, higher cutoff; null disables alerts",
            "validation_unmatched_starts_budget_per_eligible_hour": MAX_UNMATCHED_PER_HOUR,
            "validation_max_warning_fraction": MAX_WARNING_FRACTION,
            "features": MODEL_COLS, "thresholds": thresholds,
            "default_research_model": "SpO2 logistic", "deployment_validated": False,
            "test_status": "Previously inspected development benchmark; new tuning needs new external/future test data",
            "runtime": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                        "scikit_learn": sklearn.__version__}}
bundle = {"models": models, "settings": settings}
joblib.dump(bundle, OUT/"model_a.joblib", compress=3)
(OUT/"settings.json").write_text(json.dumps(settings, indent=2, allow_nan=False), encoding="utf-8")
pd.DataFrame(manifest, columns=["source","filename","url","checksum_type","checksum"]).to_csv(OUT/"source_manifest.csv",index=False)
record_audit.to_csv(OUT/"participant_splits.csv", index=False)
support.to_csv(OUT/"coverage_and_events.csv")
pd.DataFrame(weights_rows).to_csv(OUT/"logistic_coefficients.csv", index=False)
pd.concat([results, pd.DataFrame(transfer_rows)], ignore_index=True).to_csv(OUT/"metrics.csv", index=False)
print("Exported:", ", ".join(sorted(p.name for p in OUT.iterdir() if p.is_file())))

In [ ]:
def score_observed_history(bundle, times, hr_bpm, spo2_pct, name="SpO2 logistic"):
    """One 1-Hz, quality-accepted history. Score the last available second only."""
    cfg = bundle["settings"]
    if name not in bundle["models"]:
        raise ValueError(f"Unknown trained model {name}")
    n = cfg["history_samples_1hz"]
    arrays = [np.asarray(x, dtype=float) for x in (times, hr_bpm, spo2_pct)]
    if len({a.shape for a in arrays}) != 1 or any(a.ndim != 1 or len(a) < n for a in arrays):
        return {"status":"incomplete_history"}
    t, h, o = [a[-n:] for a in arrays]
    if (not np.all(np.diff(t) == 1) or not np.isfinite(h).all() or not np.isfinite(o).all()
            or (h <= 0).any() or (o <= 0).any() or (o > 100).any()):
        return {"status":"unavailable"}
    if o[-1] <= np.median(o[-cfg["baseline_samples"]:]) - cfg["drop_percentage_points"]:
        return {"status":"current_drop_already_present"}
    features = history_features(h[None,:], "hr") | history_features(o[None,:], "spo2")
    row = pd.DataFrame(features)[cfg["features"][name]]
    value = float(bundle["models"][name].predict_proba(row)[0, 1])
    cutoff = cfg["thresholds"][name]
    return {"status":"eligible", "model": name, "forecast_score": value,
            "alerts_enabled": cutoff is not None,
            "above_validation_threshold": bool(value >= cutoff) if cutoff is not None else False}

# Re-open the serialized fit and reproduce a held-out prediction from the precomputed feature row.
loaded = joblib.load(OUT/"model_a.joblib")
model_name = "SpO2 logistic"
example = test.iloc[[0]]
expected = test_scores[model_name][0]
with threadpool_limits(limits=1):
    actual = loaded["models"][model_name].predict_proba(example[MODEL_COLS[model_name]])[0,1]
assert np.isclose(actual, expected)
print("Export reload check passed.")
# The shared benchmark requires paired valid history even for the oxygen-only ablation.
# A future oxygen-only deployment contract needs its own missing-HR coverage evaluation.


**Interpreting the result.** Compare HR-only with oxygen-only and combined models before attributing any predictive value to HR. Study-specific support and alert counts matter more than a small difference in pooled AUROC. Overlapping anchors share both inputs and outcomes. Use participant-level resampling for uncertainty or collect independent subjects for a confirmatory test. Source-specific sensor averaging and measurement delay limit how the measured lead time transfers to a chest sensor. The score describes the declared relative drop and requires independent validation before entering the physiology branch.

The concern notebook is a separate current-event/rule-tuning experiment. It is not trained by treating this model's score as ground truth. Use no additional decline and current-low oxygen as distinct cases. HR quality against ECG also does not certify SpO2 accuracy.
